# Solving Poisson with scikit-fem on a FieldML Mesh

This notebook takes the bundled `unit_cube` FieldML document, converts it to a `skfem.Mesh` + `skfem.Basis` pair, and solves the Poisson problem

$$-\nabla^2 u = 1 \text{ in } \Omega, \qquad u = 0 \text{ on } \partial\Omega.$$

It demonstrates that pyfieldml slots directly into a real FEM workflow with no custom adapters.

## Install scikit-fem

```bash
pip install pyfieldml[scikit-fem]
```

In [ ]:
import numpy as np

from pyfieldml import datasets
from pyfieldml.interop.scikit_fem import to_scikit_fem

## Load a FieldML document and hand it to scikit-fem

In [ ]:
doc = datasets.load_unit_cube()
mesh, basis = to_scikit_fem(doc)
print("skfem mesh:", type(mesh).__name__)
print("skfem basis:", type(basis).__name__)
print("dofs :", basis.N)

## Assemble and solve

A textbook Poisson assembly on the unit cube. Homogeneous Dirichlet BCs on every boundary face; unit source.

In [ ]:
from skfem import BilinearForm, LinearForm, condense, solve
from skfem.helpers import dot, grad


@BilinearForm
def laplace(u, v, _):
    return dot(grad(u), grad(v))


@LinearForm
def source(v, _):
    return 1.0 * v


A = laplace.assemble(basis)
b = source.assemble(basis)
dirichlet = basis.get_dofs()
u = solve(*condense(A, b, D=dirichlet))
print("solved; max(u) =", float(u.max()))

## Inspect the solution at a few interior DOFs

In [ ]:
dof_coords = basis.doflocs.T
interior_mask = np.ones(basis.N, dtype=bool)
interior_mask[dirichlet] = False
print("interior DOF count:", int(interior_mask.sum()))
print("interior solution samples:")
for i in np.flatnonzero(interior_mask)[:5]:
    print(f"  x={dof_coords[i]}  u={u[i]:.4f}")

### Takeaway

Any FieldML Lagrange mesh is now one function call away from the full scikit-fem stack — assembly, quadrature, boundary traces, adaptive refinement. See `pyfieldml.interop.scikit_fem` for the supported topology + order table.